In [ ]:
# (1) Clone YOLOv12 repo
!git clone https://github.com/sunsmarterjie/yolov12.git
%cd yolov12

# (2) Install core dependencies manually (skip the broken wheel)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -e . --upgrade

# (3) Install compatible FlashAttention
!pip install flash-attn --no-build-isolation

# (4) Install remaining common deps
!pip install opencv-python tqdm matplotlib pyyaml tensorboard


Cloning into 'yolov12'...
remote: Enumerating objects: 1163, done.
remote: Counting objects: 100% (483/483), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 1163 (delta 325), reused 296 (delta 296), pack-reused 680 (from 2)
Receiving objects: 100% (1163/1163), 1.81 MiB | 13.21 MiB/s, done.
Resolving deltas: 100% (581/581), done.
/content/yolov12
Looking in indexes: https://download.pytorch.org/whl/cu121
Obtaining file:///content/yolov12
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.63-0.editable-py3-none-any.whl size=20235 sha256=90eb1536722d9f78ab89e70af2b80005dd93e9dd6bcd4dd1c442389596b0b3d4
  Stored in directory: /tmp/pip-ephem-wheel-cache-bnvtxaab/wheels/1c/fb/0a/30d0595ef49b9e09568

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov12n.pt")  # Should now work and auto-download weights


100%|██████████| 5.26M/5.26M [00:00<00:00, 41.1MB/s]


In [ ]:
import torch
print(torch.cuda.is_available(), torch.version.cuda)


True 12.6


In [ ]:
!pip install roboflow
from roboflow import Roboflow

# Initialize Roboflow and download
rf = Roboflow(api_key="Npr82rgYiQ5A5trJBlb8")
project = rf.workspace("capstone2025-mifho").project("military-base-object-detection")
version = project.version(15)

# Download dataset for YOLOv12 format (same as YOLOv8 format)
dataset = version.download("yolov8")  # ✅ use "yolov8" instead of "yolov12"


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Military-Base-Object-Detection-15 in yolov8:: 100%|██████████| 24198/24198 [00:08<00:00, 2688.72it/s]


FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


In [ ]:
!ls /content/Military-Base-Object-Detection-15


data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [ ]:
results = model.train(
    data="/content/Military-Base-Object-Detection-15/data.yaml",
    epochs=8,          # short since it’s just for benchmark
    imgsz=448,
    batch=16,
    device=0,
    lr0=0.002,
    project="yolov12-benchmark",
    name="exp1",
)

New https://pypi.org/project/ultralytics/8.3.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov12n.pt, data=/content/Military-Base-Object-Detection-15/data.yaml, epochs=8, time=None, patience=100, batch=16, imgsz=448, save=True, save_period=-1, cache=False, device=0, workers=8, project=yolov12-benchmark, name=exp1, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, 

100%|██████████| 755k/755k [00:00<00:00, 35.0MB/s]


Overriding model.yaml nc=80 with nc=13

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      2368  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2, 1, 2]          
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1      9344  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2, 1, 4]          
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  2    174720  ultralytics.nn.modules.block.A2C2f           [128, 128, 2, True, 4]        
  7                  -1  1    295424  ultralytic

train: Scanning /content/Military-Base-Object-Detection-15/train/labels... 8415 images, 0 backgrounds, 0 corrupt: 100%|██████████| 8415/8415 [00:04<00:00, 1949.22it/s]


train: New cache created: /content/Military-Base-Object-Detection-15/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


...

                   all       1911       4287      0.615      0.541       0.57      0.365

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        8/8      1.96G      1.333      1.309      1.253         63        448: 100%|██████████| 526/526 [02:39<00:00,  3.31it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 60/60 [00:17<00:00,  3.36it/s]


                   all       1911       4287       0.67      0.571      0.614      0.396

8 epochs completed in 0.419 hours.
Optimizer stripped from yolov12-benchmark/exp1/weights/last.pt, 5.4MB
Optimizer stripped from yolov12-benchmark/exp1/weights/best.pt, 5.4MB

Validating yolov12-benchmark/exp1/weights/best.pt...
Ultralytics 8.3.63 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12n summary (fused): 376 layers, 2,510,879 parameters, 0 gradients, 5.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 60/60 [00:19<00:00,  3.12it/s]


                   all       1911       4287       0.67      0.572      0.614      0.396
              Aircraft        127        408      0.855      0.721      0.774      0.371
            Camouflage        232        297      0.799      0.777      0.831      0.504
                 Drone         73         73      0.732      0.753      0.758       0.48
                  Fire         82        168      0.396      0.273      0.263     0.0936
               Grenade        277        502      0.634      0.552      0.616      0.453
              Hand-Gun         30         32        0.7      0.781      0.837      0.665
                 Knife        365        496      0.586      0.581      0.575      0.413
      Military-Vehicle        132        614       0.53      0.362       0.37      0.124
               Missile        233        321      0.766      0.509      0.598      0.345
                Pistol        213        287      0.616       0.69      0.708        0.6
                 Rifl

In [ ]:
metrics = model.val(data="/content/Military-Base-Object-Detection-15/data.yaml")

print("mAP@50:", metrics.box.map50)
print("mAP@75:", metrics.box.map75)
print("mAP@90:", metrics.box.maps[9])
print("Average mAP:", metrics.box.map)

print("Precision (P):", metrics.box.p)
print("Recall (R):", metrics.box.r)
F1 = 2 * metrics.box.p * metrics.box.r / (metrics.box.p + metrics.box.r)
print("F1-score:", F1)


Ultralytics 8.3.63 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12n summary (fused): 376 layers, 2,510,879 parameters, 0 gradients, 5.8 GFLOPs


val: Scanning /content/Military-Base-Object-Detection-15/valid/labels.cache... 1911 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1911/1911 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 120/120 [00:22<00:00,  5.23it/s]


                   all       1911       4287      0.668      0.571      0.614      0.396
              Aircraft        127        408      0.853      0.721      0.774      0.371
            Camouflage        232        297      0.799      0.775       0.83      0.503
                 Drone         73         73      0.731      0.753      0.757      0.481
                  Fire         82        168      0.392      0.272      0.265     0.0937
               Grenade        277        502      0.635      0.552      0.616      0.455
              Hand-Gun         30         32        0.7      0.781      0.837      0.665
                 Knife        365        496      0.582      0.579      0.575      0.413
      Military-Vehicle        132        614       0.53      0.363      0.365      0.123
               Missile        233        321      0.761      0.507      0.598      0.344
                Pistol        213        287      0.611      0.685      0.708      0.599
                 Rifl

In [ ]:
metrics = model.val(
    data="/content/Military-Base-Object-Detection-15/data.yaml",
    split="test"   # explicitly choose the test set
)

print("mAP@50:", metrics.box.map50)
print("mAP@75:", metrics.box.map75)
print("mAP@90:", metrics.box.maps[9])
print("Average mAP:", metrics.box.map)

Ultralytics 8.3.63 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)


val: Scanning /content/Military-Base-Object-Detection-15/test/labels... 1767 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1767/1767 [00:01<00:00, 1067.82it/s]


val: New cache created: /content/Military-Base-Object-Detection-15/test/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 111/111 [00:21<00:00,  5.20it/s]


                   all       1767       4130      0.642      0.542      0.573      0.352
              Aircraft        126        360      0.649      0.769      0.697      0.353
            Camouflage        230        293      0.752      0.724      0.782      0.437
                 Drone         70         79      0.533      0.443      0.443      0.234
                  Fire        110        269      0.646      0.335      0.399      0.147
               Grenade        222        365      0.717      0.647      0.722      0.527
              Hand-Gun         30         34      0.305      0.588      0.415      0.306
                 Knife         82        105        0.5      0.581      0.537      0.423
      Military-Vehicle        133        558       0.56      0.441      0.448      0.155
               Missile        232        351      0.705      0.516      0.602      0.361
                Pistol        213        280      0.735      0.596      0.716      0.527
                 Rifl

In [ ]:
metrics.box.p  # Precision (accuracy per IoU=0.5)
metrics.box.r  # Recall
print("Precision (P):", metrics.box.p)
print("Recall (R):", metrics.box.r)

Precision (P): [    0.64949     0.75215     0.53323     0.64597      0.7168     0.30485     0.49998     0.55992      0.7045     0.73491     0.71681     0.81316     0.71753]
Recall (R): [    0.76944     0.72355     0.44304     0.33457     0.64658     0.58824     0.58095     0.44086     0.51567     0.59643     0.73333    0.045627     0.63297]


In [ ]:
F1 = 2 * metrics.box.p * metrics.box.r / (metrics.box.p + metrics.box.r)
print("F1-score:", F1)


F1-score: [     0.7044     0.73757     0.48397     0.44082     0.67988     0.40158     0.53743     0.49331     0.59547     0.65847     0.72498    0.086406      0.6726]


In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("/content/yolov12-benchmark/exp1/weights/last.pt")  # or your trained best.pt

# Total parameters
total_params = sum(p.numel() for p in model.model.parameters())
print("Total parameters:", total_params)

# Trainable parameters
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_params)


Total parameters: 2522199
Trainable parameters: 0


In [ ]:
import time
from PIL import Image

# Load a sample image
img_path = "/content/Military-Base-Object-Detection-15/valid/images/018743155e71eafad2ad5c55b25703c4_jpg.rf.b5469a86db8572c2b0ccbe64be223ae0.jpg"  # or use an image from your dataset
img = Image.open(img_path)

# Warm-up
for _ in range(5):
    _ = model(img)

# Measure inference time
start = time.time()
_ = model(img)
end = time.time()

inference_time = end - start
fps = 1 / inference_time

print(f"Inference time: {inference_time:.4f} s")
print(f"FPS: {fps:.2f}")



0: 448x448 5 Grenades, 20.9ms
Speed: 2.8ms preprocess, 20.9ms inference, 2.0ms postprocess per image at shape (1, 3, 448, 448)

0: 448x448 5 Grenades, 16.1ms
Speed: 1.6ms preprocess, 16.1ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 448)

0: 448x448 5 Grenades, 16.7ms
Speed: 2.1ms preprocess, 16.7ms inference, 1.5ms postprocess per image at shape (1, 3, 448, 448)

0: 448x448 5 Grenades, 15.3ms
Speed: 1.6ms preprocess, 15.3ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 448)

0: 448x448 5 Grenades, 15.4ms
Speed: 1.8ms preprocess, 15.4ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 448)

0: 448x448 5 Grenades, 15.2ms
Speed: 1.5ms preprocess, 15.2ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 448)
Inference time: 0.0249 s
FPS: 40.19


In [ ]:
import glob

images = glob.glob("/content/Military-Base-Object-Detection-15/valid/images/*.jpg")
start = time.time()
_ = model(images[:32])  # inference on first 32 images
end = time.time()

avg_time_per_image = (end - start) / len(images[:32])
fps = 1 / avg_time_per_image
print(f"Avg inference time per image: {avg_time_per_image:.4f} s")
print(f"Approx FPS: {fps:.2f}")



0: 448x448 4 Soldiers, 3.2ms
1: 448x448 1 Grenade, 1 Pistol, 3.2ms
2: 448x448 1 Camouflage, 3.2ms
3: 448x448 1 Drone, 1 Soldier, 3.2ms
4: 448x448 (no detections), 3.2ms
5: 448x448 2 Missiles, 3.2ms
6: 448x448 1 Fire, 1 Smoke, 3.2ms
7: 448x448 (no detections), 3.2ms
8: 448x448 2 Fires, 3.2ms
9: 448x448 1 Camouflage, 3.2ms
10: 448x448 1 Camouflage, 3.2ms
11: 448x448 1 Camouflage, 1 Soldier, 3.2ms
12: 448x448 4 Soldiers, 3.2ms
13: 448x448 2 Knifes, 7 Pistols, 3.2ms
14: 448x448 1 Fire, 3.2ms
15: 448x448 1 Grenade, 3.2ms
16: 448x448 1 Knife, 1 Pistol, 3.2ms
17: 448x448 1 Grenade, 3.2ms
18: 448x448 1 Fire, 3.2ms
19: 448x448 1 Drone, 3.2ms
20: 448x448 6 Aircrafts, 3.2ms
21: 448x448 (no detections), 3.2ms
22: 448x448 1 Missile, 3.2ms
23: 448x448 2 Missiles, 3.2ms
24: 448x448 (no detections), 3.2ms
25: 448x448 1 Missile, 3.2ms
26: 448x448 2 Rifles, 3.2ms
27: 448x448 1 Camouflage, 3.2ms
28: 448x448 3 Knifes, 2 Pistols, 3.2ms
29: 448x448 (no detections), 3.2ms
30: 448x448 1 Camouflage, 3.2ms
31: